<a href="https://colab.research.google.com/github/crispythinker/coding/blob/main/Web_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install requests aiohttp beautifulsoup4 nltk transformers sentence-transformers pyttsx3 torch nest_asyncio
!apt-get install espeak-ng -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
espeak-ng is already the newest version (1.50+dfsg-10ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [11]:
!pip uninstall -y transformers
!pip install transformers==4.38.2

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 120.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 120.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.5.0
    Uninstalling huggingface_hub-1.5.0:
      Successfully uninstalled huggingface_hub-1.5.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers

In [ ]:
import requests
import aiohttp
import asyncio
import nest_asyncio
from bs4 import BeautifulSoup
from urllib.parse import quote
import nltk
from nltk import word_tokenize
from nltk.stem import WordNetLemmatizer
from transformers import pipeline
from sentence_transformers import SentenceTransformer, util
import pyttsx3
import warnings

# Fix asyncio issue in Colab
nest_asyncio.apply()

# Download nltk data
import nltk
nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('wordnet')

warnings.filterwarnings("ignore")

# Text to speech
engine = pyttsx3.init()

def speak(text):
    engine.say(text)
    engine.runAndWait()

# NLP tools
lemmatizer = WordNetLemmatizer()

# Semantic model
relevance_model = SentenceTransformer('all-MiniLM-L6-v2')

# Summarization model
summarizer = pipeline(
    "text2text-generation",
    model="sshleifer/distilbart-cnn-12-6"
)

def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(token) for token in tokens]


def summarize_text(text):

    max_input_length = 1024

    if len(text) > max_input_length:
        text = text[:max_input_length]

    result = summarizer(text, max_new_tokens=150)

    return result[0]["generated_text"]


async def fetch_content(url, session):

    try:

        async with session.get(url) as response:

            if response.status == 200:

                html = await response.text()

                soup = BeautifulSoup(html, 'html.parser')

                paragraphs = soup.find_all('p')

                content = "\n".join([p.get_text() for p in paragraphs])

                return content

    except Exception as e:

        print("Error fetching:", url)

    return None


async def search_and_summarize(query):

    tokens = word_tokenize(query)

    lemmatized = lemmatize_tokens(tokens)

    encoded_query = "+".join(lemmatized)

    encoded_query = quote(encoded_query)

    # Use DuckDuckGo instead of Google
    search_url = f"https://duckduckgo.com/html/?q={encoded_query}"

    headers = {"User-Agent": "Mozilla/5.0"}

    async with aiohttp.ClientSession() as session:

        async with session.get(search_url, headers=headers) as response:

            html = await response.text()

        soup = BeautifulSoup(html, "html.parser")

        results = soup.find_all("a", class_="result__a")

        if not results:
            print("No results found")
            return

        from urllib.parse import urlparse, parse_qs, unquote

        urls = []

        for r in results[:5]:

            raw_link = r["href"]

            parsed = urlparse(raw_link)

            query_params = parse_qs(parsed.query)

            if "uddg" in query_params:
                real_url = unquote(query_params["uddg"][0])
                urls.append(real_url)

        tasks = [fetch_content(url, session) for url in urls]

        contents = await asyncio.gather(*tasks)

        contents = [c for c in contents if c]

        if not contents:
            print("No valid content")
            return

        query_embedding = relevance_model.encode(query, convert_to_tensor=True)

        content_embeddings = relevance_model.encode(contents, convert_to_tensor=True)

        scores = util.pytorch_cos_sim(query_embedding, content_embeddings)[0]

        best_index = scores.argmax()

        best_content = contents[best_index]

        summary = summarize_text(best_content)

        print("\nSUMMARY:\n")
        print(summary)

        speak(summary)

while True:

    query = input("Enter your query (type stop to exit): ")

    if query.lower() == "stop":
        break

    asyncio.run(search_and_summarize(query))

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Enter your query (type stop to exit): how is america doing nowadays?

SUMMARY:

 85% of Americans say politically motivated violence is increasing, a rare point of agreement across the political spectrum . Many cite hostile political rhetoric, hyper-polarization, and social media–driven division as key drivers of instability they feel in their communities . Americans are not just concerned about social issues — they’re ready for change .
